# Transfer Learning

## Learning Objectives
1. Simulate frozen-backbone transfer learning to understand feature extraction vs fine-tuning.
2. Fine-tune ResNet18 on a target task in PyTorch with layer freezing and OOM-safe batching.
3. Apply BERT fine-tuning for text classification as a production NLP transfer learning example.
4. Compare data efficiency: training from scratch vs transfer learning at different dataset sizes.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Level 1: Frozen Backbone Simulation — Feature Extraction

In [ ]:
# Transfer learning: use a pretrained backbone to extract rich features,
# then train only a new classifier head on the target task.
# Simulate this without a real pretrained model.

class FrozenBackbone(nn.Module):
    """Simulate a pretrained backbone with frozen weights."""

    def __init__(self, in_features: int = 512, feature_dim: int = 128):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(in_features, 256), nn.ReLU(),
            nn.Linear(256, feature_dim), nn.ReLU(),
        )
        # Freeze backbone: no gradients computed for these layers
        for param in self.backbone.parameters():
            param.requires_grad = False

        self.head = nn.Linear(feature_dim, 10)  # Trainable classification head

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        with torch.no_grad():
            features = self.backbone(x)  # No grad through frozen layers
        return self.head(features)


model_frozen = FrozenBackbone().to(device)

# Count trainable vs total parameters
total_params = sum(p.numel() for p in model_frozen.parameters())
trainable_params = sum(p.numel() for p in model_frozen.parameters() if p.requires_grad)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters:    {total_params - trainable_params:,}")
print(f"Training overhead: {trainable_params/total_params:.1%} of full model")

# Demonstrate: gradient flows only through head
x_demo = torch.randn(8, 512, device=device)
out = model_frozen(x_demo)
loss = out.mean()
loss.backward()

backbone_grad = sum(
    0 if p.grad is None else p.grad.abs().sum().item()
    for p in model_frozen.backbone.parameters()
)
head_grad = sum(p.grad.abs().sum().item() for p in model_frozen.head.parameters())
print(f"\nBackbone gradient magnitude: {backbone_grad:.4f} (should be 0 — frozen)")
print(f"Head gradient magnitude:     {head_grad:.4f} (should be > 0 — trainable)")

## Level 2: ResNet18 Fine-Tuning in PyTorch with OOM Handling

In [ ]:
# Simulate ResNet18 fine-tuning: freeze early layers, train later layers + new head.
# In production: torchvision.models.resnet18(pretrained=True)
# Strategy 1: freeze_all -> train head only (feature extraction)
# Strategy 2: unfreeze_last_N -> fine-tune last N layers + head (full fine-tuning)

import copy
from torch.utils.data import DataLoader, TensorDataset

class ResNet18Sim(nn.Module):
    """Simulate ResNet18 architecture (without actual pretrained weights)."""

    def __init__(self, n_classes: int = 10):
        super().__init__()
        # Layer1-4 simulate early-to-late feature hierarchy
        self.layer1 = nn.Sequential(nn.Linear(512, 256), nn.ReLU(), nn.BatchNorm1d(256))
        self.layer2 = nn.Sequential(nn.Linear(256, 128), nn.ReLU(), nn.BatchNorm1d(128))
        self.layer3 = nn.Sequential(nn.Linear(128, 64), nn.ReLU(), nn.BatchNorm1d(64))
        self.layer4 = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.BatchNorm1d(32))
        self.fc = nn.Linear(32, n_classes)  # Classifier head

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(self.layer4(self.layer3(self.layer2(self.layer1(x)))))


def freeze_layers(model: nn.Module, n_unfreeze: int = 1) -> None:
    """Freeze all layers except the last n_unfreeze + classifier head."""
    all_layers = [model.layer1, model.layer2, model.layer3, model.layer4, model.fc]
    # Freeze all
    for layer in all_layers:
        for p in layer.parameters():
            p.requires_grad = False
    # Unfreeze last n_unfreeze + head
    for layer in all_layers[-n_unfreeze - 1:]:
        for p in layer.parameters():
            p.requires_grad = True


# Synthetic target-task data (small — as expected with transfer learning)
X_tl = torch.randn(500, 512, device=device)
y_tl = torch.randint(0, 10, (500,), device=device)
tr_ds = TensorDataset(X_tl[:400], y_tl[:400])
vl_ds = TensorDataset(X_tl[400:], y_tl[400:])
tr_loader = DataLoader(tr_ds, batch_size=32, shuffle=True)
vl_loader = DataLoader(vl_ds, batch_size=32)


def fine_tune(n_unfreeze: int, epochs: int = 15) -> float:
    model = ResNet18Sim(n_classes=10).to(device)
    freeze_layers(model, n_unfreeze)
    # Use different LR for head vs backbone (layer-wise LR decay)
    param_groups = [
        {'params': model.fc.parameters(), 'lr': 1e-3},
        {'params': [p for l in [model.layer1, model.layer2, model.layer3, model.layer4]
                    for p in l.parameters() if p.requires_grad], 'lr': 1e-4},
    ]
    opt = torch.optim.Adam(param_groups)
    crit = nn.CrossEntropyLoss()

    for _ in range(epochs):
        model.train()
        for xb, yb in tr_loader:
            try:
                opt.zero_grad()
                crit(model(xb), yb).backward()
                opt.step()
            except RuntimeError as exc:
                if "out of memory" in str(exc).lower():
                    torch.cuda.empty_cache()
                    print("OOM — reduce batch size or n_unfreeze")
                    continue
                raise
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for xb, yb in vl_loader:
            correct += (model(xb).argmax(1) == yb).sum().item()
            total += len(yb)
    return correct / total


print(f"{'Strategy':>25} | {'Val Acc':>9}")
print("-" * 40)
for n_unfreeze, strategy in [(0, 'head only'), (1, 'last 1 layer + head'), (4, 'all layers (full FT)')]:
    acc = fine_tune(n_unfreeze, epochs=15)
    print(f"{strategy:>25} | {acc:9.3f}")

## Real-World Example 1: BERT Fine-Tuning for Text Classification

In [ ]:
# BERT fine-tuning: attach a linear classifier to [CLS] token representation,
# freeze all but the last 2 transformer layers + classifier.
# In production: use HuggingFace Trainer; here we simulate the architecture.

class BERTSimClassifier(nn.Module):
    """Simulate BERT-base with selective layer freezing."""

    def __init__(self, n_layers: int = 12, d_model: int = 768, n_classes: int = 2,
                 unfreeze_last: int = 2):
        super().__init__()
        # Simulate 12 transformer layers
        self.transformer_layers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(d_model, d_model * 4), nn.GELU(),
                nn.Linear(d_model * 4, d_model), nn.LayerNorm(d_model)
            ) for _ in range(n_layers)
        ])
        self.classifier = nn.Linear(d_model, n_classes)

        # Freeze all transformer layers
        for layer in self.transformer_layers:
            for p in layer.parameters():
                p.requires_grad = False
        # Unfreeze last `unfreeze_last` layers
        for layer in self.transformer_layers[-unfreeze_last:]:
            for p in layer.parameters():
                p.requires_grad = True

    def forward(self, cls_repr: torch.Tensor) -> torch.Tensor:
        """cls_repr: (batch, d_model) — simulated [CLS] token from last layer."""
        return self.classifier(cls_repr)


# Simulate [CLS] token representations from BERT preprocessing
d_model = 768
N_text = 1000
cls_train = torch.randn(800, d_model, device=device)
cls_val = torch.randn(200, d_model, device=device)
y_text_tr = torch.randint(0, 2, (800,), device=device)
y_text_vl = torch.randint(0, 2, (200,), device=device)

bert_model = BERTSimClassifier(n_layers=12, d_model=d_model, n_classes=2, unfreeze_last=2).to(device)

total_p = sum(p.numel() for p in bert_model.parameters())
trainable_p = sum(p.numel() for p in bert_model.parameters() if p.requires_grad)
print(f"BERT-sim: {total_p/1e6:.1f}M total, {trainable_p/1e6:.1f}M trainable "
      f"({trainable_p/total_p:.1%})")

bert_loader_tr = DataLoader(TensorDataset(cls_train, y_text_tr), batch_size=32, shuffle=True)
bert_opt = torch.optim.AdamW(bert_model.parameters(), lr=2e-5, weight_decay=0.01)
bert_crit = nn.CrossEntropyLoss()

for epoch in range(10):
    bert_model.train()
    for xb, yb in bert_loader_tr:
        bert_opt.zero_grad()
        bert_crit(bert_model(xb), yb).backward()
        nn.utils.clip_grad_norm_(bert_model.parameters(), 1.0)
        bert_opt.step()

bert_model.eval()
with torch.no_grad():
    acc_bert = (bert_model(cls_val).argmax(1) == y_text_vl).float().mean().item()
print(f"BERT fine-tune val accuracy: {acc_bert:.3f}")
print("Production tip: use lr=2e-5, AdamW, warmup 10% steps, linear decay.")

## Real-World Example 2: Layer-wise Learning Rate Decay

In [ ]:
# Layer-wise LR decay (LLRD): lower layers get smaller learning rates.
# Rationale: lower layers contain general features; higher layers are task-specific.
# LLRD prevents catastrophic forgetting of pretrained representations.

def build_llrd_optimizer(
    model: nn.Module,
    base_lr: float = 2e-5,
    decay_factor: float = 0.9,
) -> torch.optim.AdamW:
    """Create AdamW with layer-wise LR decay.
    
    Layer 0 (lowest): lr = base_lr * decay_factor^n_layers
    Layer n (head):   lr = base_lr
    """
    n_layers = len(model.transformer_layers)
    param_groups = []

    # Classifier head: full learning rate
    param_groups.append({
        'params': model.classifier.parameters(),
        'lr': base_lr,
        'layer': 'head',
    })

    # Transformer layers: exponentially decaying LR from top to bottom
    for layer_idx, layer in enumerate(reversed(model.transformer_layers)):
        lr = base_lr * (decay_factor ** (layer_idx + 1))
        param_groups.append({
            'params': [p for p in layer.parameters() if p.requires_grad],
            'lr': lr,
            'layer': f'transformer_{n_layers - layer_idx - 1}',
        })

    # Filter out empty param groups
    param_groups = [pg for pg in param_groups if len(list(pg['params'])) > 0 or
                    pg.get('params') and len(pg['params']) > 0]

    return torch.optim.AdamW(param_groups, weight_decay=0.01)


bert_llrd = BERTSimClassifier(n_layers=12, d_model=768, n_classes=2, unfreeze_last=12).to(device)
llrd_opt = build_llrd_optimizer(bert_llrd, base_lr=2e-5, decay_factor=0.9)

print("Layer-wise learning rates (head -> bottom):")
for pg in llrd_opt.param_groups[:6]:  # Show first 6
    print(f"  {pg.get('layer', 'unknown'):>20}: lr = {pg['lr']:.2e}")
print(f"  ... (showing 6/{len(llrd_opt.param_groups)} groups)")

# Train a few steps
for xb, yb in bert_loader_tr:
    llrd_opt.zero_grad()
    bert_crit(bert_llrd(xb), yb).backward()
    nn.utils.clip_grad_norm_(bert_llrd.parameters(), 1.0)
    llrd_opt.step()
    break

print("\nLLRD training step completed. Lower layers updated very slowly.")

## Real-World Example 3: Data Efficiency — Scratch vs Transfer Learning

In [ ]:
# Transfer learning shines when labeled data is scarce.
# Compare: training from scratch vs using pretrained features
# at different training set sizes.

class FromScratchModel(nn.Module):
    """Small network trained from scratch on target task."""
    def __init__(self, n_classes: int = 10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(512, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, n_classes),
        )
    def forward(self, x): return self.net(x)


class TransferModel(nn.Module):
    """Pretrained backbone (frozen) + small trainable head."""
    def __init__(self, n_classes: int = 10):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(512, 256), nn.ReLU(), nn.Linear(256, 128), nn.ReLU()
        )
        # Freeze backbone (simulate pretrained)
        for p in self.backbone.parameters():
            p.requires_grad = False
        self.head = nn.Linear(128, n_classes)

    def forward(self, x):
        with torch.no_grad():
            feats = self.backbone(x)
        return self.head(feats)


def train_eval(model, n_train, epochs=20, batch=32):
    X_all = torch.randn(n_train + 200, 512, device=device)
    y_all = torch.randint(0, 10, (n_train + 200,), device=device)
    tr = DataLoader(TensorDataset(X_all[:n_train], y_all[:n_train]), batch_size=batch, shuffle=True)
    vl = DataLoader(TensorDataset(X_all[n_train:], y_all[n_train:]), batch_size=batch)
    opt = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
    crit = nn.CrossEntropyLoss()
    for _ in range(epochs):
        model.train()
        for xb, yb in tr:
            opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for xb, yb in vl:
            correct += (model(xb).argmax(1) == yb).sum().item(); total += len(yb)
    return correct / total


print(f"{'N Train':>9} | {'Scratch':>9} | {'Transfer':>10} | {'Gain':>6}")
print("-" * 44)
for n_train in [50, 100, 200, 500, 1000]:
    acc_scratch = train_eval(FromScratchModel().to(device), n_train)
    acc_transfer = train_eval(TransferModel().to(device), n_train)
    print(f"{n_train:>9} | {acc_scratch:9.3f} | {acc_transfer:10.3f} | "
          f"{acc_transfer - acc_scratch:+6.3f}")

print("\nTransfer learning advantage is largest at small N; diminishes as data grows.")